# TEP fault detection: Colab runner

Runs `tep_pipeline.py` from this repo on the dataset stored in your Google Drive.

1. **Setup** mounts Drive and fetches the latest code from GitHub (re-run it to pick up changes).
2. **Settings**: point `DATA_DIR` at the Drive folder holding the four TEP CSVs.
3. **Train** runs the model grid. Finished models are saved to Drive, so if Colab disconnects, just run it again.
4. **Results** shows the validation tables. **Final test** runs once, at the very end.
5. **Download** zips the website files.

In [ ]:
# Setup: mount Drive and get the latest code
from google.colab import drive
drive.mount("/content/drive")

%cd /content
!git clone -q https://github.com/Fang099/tep-fault-detection.git 2>/dev/null || git -C /content/tep-fault-detection pull -q
%cd /content/tep-fault-detection
!git log -1 --format="Code version: %h (%cr) %s"

In [ ]:
# Settings
DATA_DIR = "/content/drive/MyDrive/TEP"   # folder with the 4 CSVs (results are saved here too)
RUNS_PER_FAULT = 100                       # 0 = all 500 runs per fault

In [ ]:
# Train and validate every model (skips models already finished)
!python -u tep_pipeline.py --data-dir "{DATA_DIR}" --work-dir /content/dataset --runs-per-fault {RUNS_PER_FAULT}

In [ ]:
# Results: validation metrics for every model
import json
from pathlib import Path
import pandas as pd

out = Path(DATA_DIR) / ("outputs_full" if RUNS_PER_FAULT == 0 else f"outputs_{RUNS_PER_FAULT}runs") / "web_data"
manifest = json.loads((out / "manifest.json").read_text())

for task, key in [("detection", "recall"), ("diagnosis", "macro_F1")]:
    table = pd.DataFrame([{"model": m["label"], "nonzero coefs": m["coef_nonzero"], **m["val"]}
                          for m in manifest[task]["models"]]).set_index("model")
    print(f"\n{task.upper()}  (best: {manifest[task]['best']}, rule: {manifest[task]['selection_rule']})")
    display(table.drop(columns=[c for c in ("TN", "FP", "FN", "TP") if c in table]).sort_values(key, ascending=False).round(4))

In [ ]:
# Final test: run ONCE, after you are done choosing models
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    !python -u tep_pipeline.py --data-dir "{DATA_DIR}" --work-dir /content/dataset --runs-per-fault {RUNS_PER_FAULT} --final-test

In [ ]:
# Download the website files as a zip
import shutil
from google.colab import files
shutil.make_archive("/content/web_data", "zip", out)
files.download("/content/web_data.zip")